# BLB Hebrew Pronunciation Audio Downloader + Analyser + Splitter

**Cell 1** — Config  
**Cell 2** — Setup (folders, cache)  
**Cell 3** — Helpers  
**Cell 4** — Download loop (resume-safe)  
**Cell 5** — Silence analysis → CSV  
**Cell 6** — Split / reformat audio with ffmpeg  

Re-running any cell is safe — already-done work is skipped.

## Cell 1 — Config

In [ ]:
START      = 1        # first Strong's number
END        = 100      # last Strong's number (inclusive)

MUSIC_DIR   = "music"        # raw downloaded audio
HTML_DIR    = "strongs_h"    # cached lexicon HTML pages
SPLIT_DIR   = "strongs_p"    # final split/reformatted audio

HASH_CACHE_FILE   = "blb_pronunc_hashes.json"
SILENCE_CSV_FILE  = "blb_silence_analysis.csv"

DELAY_OK    = 1.5   # seconds between successful requests
DELAY_RETRY = 30    # base backoff seconds on rate-limit / timeout
MAX_RETRIES = 5

# ffmpeg silence detection thresholds
SILENCE_DB       = -35    # dB threshold — adjust if too many/few gaps detected
SILENCE_MIN_DUR  = 0.5    # minimum gap duration in seconds to count as silence

## Cell 2 — Setup

In [ ]:
import os, json, time, re, shutil, subprocess
import urllib.request, urllib.error, urllib.parse
from pathlib import Path
import pandas as pd

for d in (MUSIC_DIR, HTML_DIR, SPLIT_DIR):
    Path(d).mkdir(parents=True, exist_ok=True)
    print(f"✅ Folder ready: {Path(d).resolve()}")

# ── hash cache ─────────────────────────────────────────────────────────────
cache_path = Path(HASH_CACHE_FILE)
if cache_path.exists():
    with open(cache_path, "r", encoding="utf-8") as f:
        hash_cache: dict = json.load(f)
    print(f"📦 Loaded hash cache ({len(hash_cache)} entries)")
else:
    hash_cache = {}
    print("📦 Fresh hash cache")

def save_cache():
    with open(cache_path, "w", encoding="utf-8") as f:
        json.dump(hash_cache, f, indent=2)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

try:
    subprocess.run(["ffmpeg", "-version"], capture_output=True, check=True)
    print("✅ ffmpeg found")
except FileNotFoundError:
    print("❌ ffmpeg not found — install it before running cells 5 and 6")

## Cell 3 — Helpers

In [ ]:
# ── network ────────────────────────────────────────────────────────────────

def fetch_html(url: str, retries: int = MAX_RETRIES) -> str | None:
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=20) as resp:
                return resp.read().decode("utf-8", errors="replace")
        except urllib.error.HTTPError as e:
            if e.code in (429, 503):
                wait = DELAY_RETRY * attempt
                print(f"    ⏳ Rate-limited ({e.code}), waiting {wait}s (attempt {attempt}/{retries})...")
                time.sleep(wait)
            else:
                print(f"    ❌ HTTP {e.code} — {url}")
                return None
        except (urllib.error.URLError, TimeoutError, OSError) as e:
            wait = DELAY_RETRY * attempt
            print(f"    ⏳ Network error ({e}), waiting {wait}s (attempt {attempt}/{retries})...")
            time.sleep(wait)
    print(f"    ❌ Gave up after {retries} attempts: {url}")
    return None


def extract_pronunc_hash(html: str) -> str | None:
    m = re.search(r'data-pronunc="([^"]+)"', html)
    return m.group(1) if m else None


def audio_url(skin_hash: str) -> str:
    return (
        "https://www.blueletterbible.org/lang/lexicon/"
        f"lexPronouncePlayer.cfm?skin={skin_hash}"
    )

import requests
def download_audio(url: str, dest_dir: Path, strong_key: str,
                   retries: int = MAX_RETRIES) -> Path | None:
    session = requests.Session()
    session.headers.update(HEADERS)
    
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, timeout=30, stream=True)
            
            if resp.status_code in (429, 503):
                wait = DELAY_RETRY * attempt
                print(f"    ⏳ Rate-limited ({resp.status_code}), waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            
            # get filename from Content-Disposition or fallback
            cd = resp.headers.get("Content-Disposition", "")
            fname_match = re.search(
                r'filename\*?=(?:UTF-8\'\'|"?)([^";\r\n]+)', cd, re.I
            )
            if fname_match:
                # server sends full server-side path like /mnt/nginx-server/.../h0001.mp3
                # take only the basename, just like curl -J does
                filename = Path(urllib.parse.unquote(fname_match.group(1).strip('"'))).name
            else:
                ct = resp.headers.get("Content-Type", "")
                ext = ".mp3" if "mpeg" in ct else ".ogg" if "ogg" in ct else ".audio"
                filename = f"{strong_key}{ext}"
            
            dest = dest_dir / filename
            with open(dest, "wb") as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            return dest
            
        except requests.exceptions.HTTPError as e:
            print(f"    ❌ HTTP error: {e}")
            return None
        except requests.exceptions.RequestException as e:
            wait = DELAY_RETRY * attempt
            print(f"    ⏳ Request error ({e}), waiting {wait}s (attempt {attempt}/{retries})...")
            time.sleep(wait)
    
    print(f"    ❌ Gave up after {retries} attempts: {strong_key}")
    return None

# ── ffmpeg helpers ─────────────────────────────────────────────────────────

def detect_silences(audio_path: Path,
                    noise_db: int = SILENCE_DB,
                    min_dur: float = SILENCE_MIN_DUR) -> list[dict]:
    """
    Run ffmpeg silencedetect and return a list of dicts:
      { 'silence_start': float, 'silence_end': float, 'silence_dur': float }
    """
    cmd = [
        "ffmpeg", "-i", str(audio_path),
        "-af", f"silencedetect=noise={noise_db}dB:d={min_dur}",
        "-f", "null", "-"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    output = result.stderr   # silencedetect writes to stderr
    starts = re.findall(r"silence_start:\s*([\d.]+)", output)
    ends   = re.findall(r"silence_end:\s*([\d.]+)",   output)
    durs   = re.findall(r"silence_duration:\s*([\d.]+)", output)
    return [
        {"silence_start": float(s), "silence_end": float(e), "silence_dur": float(d)}
        for s, e, d in zip(starts, ends, durs)
    ]


def get_audio_duration(audio_path: Path) -> float | None:
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(audio_path)
    ]
    r = subprocess.run(cmd, capture_output=True, text=True)
    try:
        return float(r.stdout.strip())
    except ValueError:
        return None


print("✅ Helpers ready")

## Cell 4 — Download loop (resume-safe)

Re-run freely — cached HTML and existing audio files are skipped.

In [ ]:
skipped_no_hash = []
skipped_dl_fail = []
downloaded      = []
already_had     = []

html_root  = Path(HTML_DIR)
music_root = Path(MUSIC_DIR)
START=101
END=500
from tqdm.notebook import tqdm
for num in tqdm(range(START, END + 1)):
    key     = f"h{num}"
    html_fp = html_root / f"{key}.html"

    # ── STEP 1: get pronunc hash ───────────────────────────────────────────
    # Priority: in-memory cache → saved HTML on disk → live fetch
    if key in hash_cache:
        skin = hash_cache[key]
    elif html_fp.exists():
        with open(html_fp, "r", encoding="utf-8") as f:
            cached_html = f.read()
        skin = extract_pronunc_hash(cached_html)
        print(f"[{num}/{END}] 📄 {key}: hash from saved HTML")
        hash_cache[key] = skin
        save_cache()
    else:
        lex_url = f"https://www.blueletterbible.org/lexicon/{key}/kjv/wlc/0-1/"
        print(f"[{num}/{END}] 🌐 {key}: fetching {lex_url}")
        html = fetch_html(lex_url)
        if html is None:
            print(f"  ⚠️  Could not fetch page for {key}, skipping")
            skipped_dl_fail.append(key)
            continue
        # save HTML to disk
        with open(html_fp, "w", encoding="utf-8") as f:
            f.write(html)
        skin = extract_pronunc_hash(html)
        hash_cache[key] = skin
        save_cache()
        time.sleep(DELAY_OK)

    if not skin:
        print(f"[{num}/{END}] ⚪ {key}: no data-pronunc found — no audio")
        skipped_no_hash.append(key)
        continue

    # ── STEP 2: skip if any audio file for this key already exists ─────────
    existing = list(music_root.glob(f"{key}.*"))
    if existing:
        print(f"[{num}/{END}] ✔️  {key}: already have {existing[0].name}")
        already_had.append(key)
        continue

    # ── STEP 3: download audio ─────────────────────────────────────────────
    aurl = audio_url(skin)
    print(f"[{num}/{END}] 🎵 {key}: downloading...")
    dest = download_audio(aurl, music_root, key)
    if dest:
        print(f"  ✅ Saved → {dest.name}")
        downloaded.append(str(dest))
    else:
        skipped_dl_fail.append(key)

    time.sleep(DELAY_OK)

print()
print("=" * 50)
print(f"✅ Newly downloaded : {len(downloaded)}")
print(f"✔️  Already had      : {len(already_had)}")
print(f"⚪ No audio         : {len(skipped_no_hash)}  {skipped_no_hash[:10]}")
print(f"❌ Failed           : {len(skipped_dl_fail)}  {skipped_dl_fail}")

In [ ]:
hash_cache['h1']='81FC0FAC3BD04B2CF6239CE007367F3051593AAC0C4C65080B1FC883226ED707F2619F9063F690D1D477F592E826C78083A6DA2CDAAC'

In [ ]:
hash_cache

## Cell 5 — Silence analysis → CSV

Runs `ffmpeg silencedetect` on every file in `music/`.  
**One row per silence gap** — so a file with 3 gaps produces 3 rows (plus metadata columns shared across all rows for that file).  
Re-running skips already-analysed keys.

### Inferred audio structure
```
Single-entry, one pronunciation variant:
  [H-number spoken]  GAP-0(long)  [pronunc]  (end)

Single-entry, two pronunciation variants:
  [H-number spoken]  GAP-0(long)  [pronunc A]  GAP-1(short)  [pronunc B]  (end)

Two lexicon entries:
  [H-number spoken]  GAP-0(long)  [pronunc 1A]  GAP-1(short?)  [pronunc 1B?]
  GAP-2(long)  ["second entry"]  GAP-3(long)  [pronunc 2A]  GAP-4(short?)  [pronunc 2B?]  (end)
```
Gap index (`silence_index`, 0-based) tells you position in the file.  
Longer gaps mark structural boundaries; shorter gaps separate pronunciation variants.

In [ ]:
csv_path   = Path(SILENCE_CSV_FILE)
music_root = Path(MUSIC_DIR)

# load existing CSV to skip already-analysed keys
if csv_path.exists():
    df_existing = pd.read_csv(csv_path)
    already_analysed = set(df_existing["key"].unique())
    print(f"📊 Loaded existing CSV ({len(df_existing)} rows, {len(already_analysed)} keys)")
else:
    df_existing = pd.DataFrame()
    already_analysed = set()
    print("📊 No existing CSV — starting fresh")

new_rows = []

# collect all audio files, sorted numerically
def sort_key(p):
    m = re.search(r"\d+", p.stem)
    return int(m.group()) if m else 0

audio_files = []
for ext in ("mp3", "ogg", "audio"):
    audio_files += sorted(music_root.glob(f"h*.{ext}"), key=sort_key)

from tqdm.notebook import tqdm
for audio_fp in tqdm(audio_files):
    m = re.search(r"(h\d+)", audio_fp.stem, re.I)
    if not m:
        print(f"⚠️  Can't parse key from: {audio_fp.name} — skipping")
        continue
    key = m.group(1).lower()

    if key in already_analysed:
        print(f"✔️  {key}: already analysed")
        continue

    print(f"🔍 {key}: analysing {audio_fp.name}...", end=" ", flush=True)
    total_dur = get_audio_duration(audio_fp)
    silences  = detect_silences(audio_fp)
    print(f"{len(silences)} gap(s)  (total {total_dur:.2f}s)")

    base_row = {
        "key":           key,
        "filename":      audio_fp.name,
        "total_dur":     total_dur,
        "silence_count": len(silences),
    }

    if not silences:
        # one row with nulls for silence columns
        new_rows.append({**base_row,
                         "silence_index": None,
                         "silence_start": None,
                         "silence_end":   None,
                         "silence_dur":   None})
    else:
        for idx, s in enumerate(silences):
            new_rows.append({**base_row,
                             "silence_index": idx,
                             "silence_start": s["silence_start"],
                             "silence_end":   s["silence_end"],
                             "silence_dur":   s["silence_dur"]})

# save
if new_rows:
    df_new = pd.DataFrame(new_rows)
    df_all = pd.concat([df_existing, df_new], ignore_index=True)
    df_all.to_csv(csv_path, index=False)
    print(f"\n💾 Saved {len(df_all)} total rows → {csv_path}")
else:
    df_all = df_existing
    print("\nNo new files to analyse.")

# summary table
if not df_all.empty:
    summary = (
        df_all.groupby("key")["silence_count"]
        .first()
        .value_counts()
        .sort_index()
        .rename_axis("gap_count")
        .rename("files")
    )
    print("\nSilence-gap distribution:")
    print(summary.to_string())

In [ ]:
df_all = pd.read_csv("blb_silence_analysis.csv")

for n_gaps in [3, 6]:
    sample = df_all[df_all["silence_count"] == n_gaps]["key"].iloc[0]
    rows = df_all[df_all["key"] == sample].dropna(subset=["silence_start"])
    print(f"\n{sample} ({n_gaps} gaps), total_dur={rows.iloc[0]['total_dur']:.2f}s")
    for _, r in rows.iterrows():
        print(f"  gap {int(r['silence_index'])}: {r['silence_start']:.2f}s → {r['silence_end']:.2f}s  (dur={r['silence_dur']:.2f}s)")

## Cell 6 — Split & reformat with ffmpeg

### Segment assignment logic

```
speech_seg[0]           → ALWAYS DROP  (speaker says the Strong's number)
speech_seg[1]           → entry-1, pronunciation A          → h<N>.mp3
speech_seg[2] (if any)  → depends on gap before it:
    SHORT gap            → entry-1, pronunciation B (concat with A) → h<N>.mp3
    LONG gap             → entry-2 preamble ("second entry") → DROP
speech_seg[3] (if any)  → entry-2, pronunciation A           → h<N>-2.mp3
...and so on.
```

**Short vs long** gap heuristic: threshold = 70% of median gap duration per file.  
Gaps >= threshold are "long" (structural boundary); below are "short" (variant separator).

Output in `strongs_p/`:
- `h1.mp3`   — entry 1 (all pronunciation variants concatenated if >1)
- `h1-2.mp3` — entry 2 (if a second lexicon entry exists)

Re-running skips already-split keys.

In [ ]:
import statistics

split_root = Path(SPLIT_DIR)
music_root = Path(MUSIC_DIR)
csv_path   = Path(SILENCE_CSV_FILE)

if not csv_path.exists():
    raise RuntimeError("Run Cell 5 first to generate the silence CSV.")

df_all = pd.read_csv(csv_path)


# ── ffmpeg extract/concat helpers ──────────────────────────────────────────

def ffmpeg_extract(src: Path, start: float, end: float, dest: Path):
    """Cut [start, end) seconds from src → dest (mp3)."""
    cmd = [
        "ffmpeg", "-y",
        "-i", str(src),
        "-ss", f"{start:.6f}",
        "-t",  f"{end - start:.6f}",
        "-c:a", "libmp3lame", "-q:a", "2",
        str(dest)
    ]
    subprocess.run(cmd, capture_output=True, check=True)


def ffmpeg_concat(parts: list[Path], dest: Path):
    """Concatenate audio segments into dest."""
    if len(parts) == 1:
        shutil.copy(parts[0], dest)
        return
    list_file = dest.parent / f"_cl_{dest.stem}.txt"
    with open(list_file, "w") as f:
        for p in parts:
            f.write(f"file '{p.resolve()}'\n")
    cmd = [
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0",
        "-i", str(list_file),
        "-c:a", "libmp3lame", "-q:a", "2",
        str(dest)
    ]
    subprocess.run(cmd, capture_output=True, check=True)
    list_file.unlink(missing_ok=True)


# ── per-key splitting ──────────────────────────────────────────────────────

def split_key(key: str, df: pd.DataFrame) -> list[Path]:
    rows = df[df["key"] == key].sort_values("silence_index")
    if rows.empty:
        return []

    src_name  = rows.iloc[0]["filename"]
    src       = music_root / src_name
    if not src.exists():
        print(f"  ⚠️  Source not found: {src}")
        return []

    total_dur  = float(rows.iloc[0]["total_dur"])
    n_silences = int(rows.iloc[0]["silence_count"])

    # ── no silences: copy whole file ───────────────────────────────────────
    if n_silences == 0:
        out = split_root / f"{key}.mp3"
        shutil.copy(src, out)
        return [out]

    sil_rows = rows.dropna(subset=["silence_start"]).sort_values("silence_index")
    silences = list(zip(
        sil_rows["silence_start"].astype(float),
        sil_rows["silence_end"].astype(float),
        sil_rows["silence_dur"].astype(float),
    ))

    # ── short vs long threshold ────────────────────────────────────────────
    durs = [d for _, _, d in silences]
    threshold = statistics.median(durs) * 0.7 if len(durs) > 1 else durs[0]

    def is_long(dur):
        return dur >= threshold

    # ── build speech segments from gap boundaries ──────────────────────────
    # alternating: [speech][silence][speech][silence]...[speech]
    boundaries = [0.0]
    for s_start, s_end, _ in silences:
        boundaries.extend([s_start, s_end])
    boundaries.append(total_dur)

    speech_segs = []          # (start, end)
    gap_is_long = []          # True/False for the gap BEFORE speech_segs[i+1]

    for i in range(0, len(boundaries) - 1, 2):
        seg_s = boundaries[i]
        seg_e = boundaries[i + 1]
        if seg_e - seg_s > 0.05:   # ignore tiny slivers
            speech_segs.append((seg_s, seg_e))

    # gap_is_long[i] = True means there is a LONG silence between
    # speech_segs[i] and speech_segs[i+1]
    for _, _, dur in silences:
        gap_is_long.append(is_long(dur))

    # ── assign speech segments to entries ─────────────────────────────────
    # speech_segs[0] = H-number spoken → always DROP
    #
    # Walk speech_segs[1:].
    # LONG gap before current seg:
    #   - if current_entry is empty  → this is pronunc A of the next entry
    #   - if current_entry has items → close it, check if this new seg is
    #     a "second entry" preamble (it will be followed by another LONG gap)
    #     or pronunc A directly.
    # SHORT gap before current seg:
    #   → pronunciation variant B (or C) of the current entry

    entries: list[list[tuple]]  = []   # list of entries; each entry = list of (start,end) segs
    current_entry: list[tuple]  = []
    drop_next = False   # True when the next seg is a spoken preamble to drop

    for seg_i in range(1, len(speech_segs)):
        seg = speech_segs[seg_i]
        gap_before_long = gap_is_long[seg_i - 1] if (seg_i - 1) < len(gap_is_long) else True

        if drop_next:
            # this is the "second entry" spoken preamble — skip it
            drop_next = False
            continue

        if gap_before_long:
            # close current entry (if open)
            if current_entry:
                entries.append(current_entry)
                current_entry = []

            # peek at the gap AFTER this seg to decide if it's a preamble or pronunc
            # If the gap AFTER this seg is also LONG, it is a spoken label → drop it
            gap_after_long = (
                gap_is_long[seg_i] if seg_i < len(gap_is_long) else False
            )
            if gap_after_long:
                # this segment is a spoken preamble ("second entry") → drop
                drop_next = False  # the NEXT seg starts a new entry
            else:
                # this segment is pronunciation A of a new entry
                current_entry.append(seg)
        else:
            # short gap → pronunciation variant, stays in current entry
            current_entry.append(seg)

    if current_entry:
        entries.append(current_entry)

    # ── extract segments and save ──────────────────────────────────────────
    saved   = []
    tmp_dir = split_root / f"_tmp_{key}"
    tmp_dir.mkdir(exist_ok=True)

    for entry_idx, segs in enumerate(entries):
        out_name = f"{key}.mp3" if entry_idx == 0 else f"{key}-{entry_idx + 1}.mp3"
        out_path = split_root / out_name

        if len(segs) == 1:
            ffmpeg_extract(src, segs[0][0], segs[0][1], out_path)
        else:
            parts = []
            for vi, (vs, ve) in enumerate(segs):
                tmp = tmp_dir / f"v{vi}.mp3"
                ffmpeg_extract(src, vs, ve, tmp)
                parts.append(tmp)
            ffmpeg_concat(parts, out_path)

        saved.append(out_path)

    shutil.rmtree(tmp_dir, ignore_errors=True)
    return saved


# ── run over all analysed keys ─────────────────────────────────────────────

keys_in_csv = sorted(
    df_all["key"].unique(),
    key=lambda k: int(re.search(r"\d+", k).group())
)
from tqdm.notebook import tqdm
for key in tqdm(keys_in_csv):
    if (split_root / f"{key}.mp3").exists():
        print(f"✔️  {key}: already split")
        continue

    try:
        saved = split_key(key, df_all)
        if saved:
            print(f"✅ {key}: → {', '.join(p.name for p in saved)}")
        else:
            print(f"⚠️  {key}: nothing produced")
    except subprocess.CalledProcessError as e:
        stderr_tail = e.stderr[-300:] if e.stderr else ""
        print(f"❌ {key}: ffmpeg error — {stderr_tail}")
    except Exception as e:
        print(f"❌ {key}: {e}")

print("\n🏁 Done.")